IPython Notebook file run and tested in T4 runtime of google colab

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [2]:
# imports

from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import pandas as pd
import gradio as gr
import time

In [3]:
# HF login and model

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

LLAMA = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

In [4]:
# Quantize and tokenize model

def load_model(model_name):
  """
  Configure quantization method, tokenize the model and convert pad tokens to eos tokens.
  Apply quantization to chosen model.
  return the tokenizer and quantized model.
  """

  quant_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=torch.bfloat16,
      bnb_4bit_quant_type='nf4'
  )

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = tokenizer.eos_token
  model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=quant_config)
  return tokenizer, model

In [ ]:
# load model

tokenizer, model = load_model(LLAMA)

In [6]:
# Find unique records

def add_batch(batch, records, unique_keys):
  """
  Find duplicate dictionaries in batch list.
  Append only unique dictionaries / records from the batch.
  """

  for record in batch:
    key = json.dumps(record, sort_keys=True)

    if key not in unique_keys:
      unique_keys.add(key)
      records.append(record)

In [7]:
# llm call

def generate_response(tokenizer, model, messages, max_new_tokens=2048):
  """
  applies chat template to the tokenizer and converts it into tensor using pytorch.
  places the tensor in gpu using .to(cuda).
  define output parameters with temperature and do_sample for randomness.
  slice content after the input till the end to get only output.
  skip special tokens like <eos> in output and return output
  """

  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

  outputs = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)

  return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

In [8]:
# estimate tokens for 1 record to calculate batch size

def estimate_tokens_per_record(tokennizer, model, base_prompt):
  """
  Generate small set of sample records to calculate the output tokens for 1 record
  """

  sample_prompt = (
      base_prompt +
      "\n\n Generate exactly 3 records."
      "\n return only valid JSON array."
      "\n No markdown"
      "\n No explanations"
  )

  messages = [{
      "role":"user", "content":sample_prompt
  }]

  response = generate_response(tokenizer, model, messages, max_new_tokens=1024)
  print(response)

  try:
    data = json.loads(response)
    print(type(data))
    print(len(data))

    if not isinstance(data, list) or len(data) == 0:
      return 100

    output_tokens = len(tokenizer.encode(response))
    return max(1, output_tokens/len(data))

  #except Exception:
    #return 100

  except Exception as e:
    print("ERROR:", e)
    print(response)
    return 100

In [9]:
# calculate batch size

def calculate_batch_size(avg_tokens_per_record, max_output_tokens=2048, safety_margin=200):
  """
  calculates available tokens after negating safety_margin to account estimate inconsistencies.
  Find batch size to safely generate responses in batches efficiently.
  """

  available = max_output_tokens - safety_margin

  batch_size = int(available // avg_tokens_per_record)

  return max(1,batch_size)

In [10]:
def get_system_prompt(batch_size):

  system_prompt = f"""
  You are a synthetic data generator.
  Your task is to generate synthetic data based on the user's requirement.
  Do not perform any other task other than generating synthetic data.
  Generate EXACTLY {batch_size} records.

  Requirements:
  - Return ONLY valid JSON.
  - Return a JSON array.
  - Array length must be exactly {batch_size}.
  - No markdown.
  - No explanations.

  example : generate employee data

  [
    {{
      "name" : "kevin",
      "age" : 25,
      "salary" : 25000
    }},

    {{
      "name" : "wilfred",
      "age" : 27,
      "salary" : 55000
    }}

  ]
  """

  return system_prompt

In [11]:
def get_user_prompt(domain, description):
  user_prompt = f"""
  generate data in {domain} following : {description}.
  """

  return user_prompt

In [12]:
# Function that uses other helper functions to generate synthetic data records

def generate_records(tokenizer, model,domain, description, count, max_output_tokens=2048):
  """
  Repeadtedly calls the llm until it creates the required number of unique synthetic data
  or until the maximum attempts is crossed.
  """

  records = []
  unique_keys = set()

  avg_tokens = estimate_tokens_per_record(tokenizer, model, description)

  batch_limit = calculate_batch_size(avg_tokens, max_output_tokens=max_output_tokens)

  print(f"Estimated tokens/record={avg_tokens:.1f}, "
        f"batch_size={batch_limit}")

  attempts = 0
  max_attempts = count * 5

  while len(records) < count and attempts < max_attempts:

    remaining = count - len(records)
    batch_size = min(remaining, batch_limit)     # if remaining is low generates only those records and not full batch

    messages = [
        {"role":"system", "content": get_system_prompt(batch_size)},
        {"role":"user", "content": get_user_prompt(domain, description)}
    ]

    response = generate_response(tokenizer, model, messages, max_new_tokens=max_output_tokens)

    try:
      batch = json.loads(response)

      if not isinstance(batch, list):
        attempts += 1
        print("Response is not a list, retrying...")
        continue

      if len(batch) == 0:
          attempts += 1
          print("Empty batch, retrying...")
          continue

      add_batch(batch, records, unique_keys)
      print(f'collected {len(records)}/{count} unique records')
      attempts+=1

    except json.JSONDecodeError:
      print("Invalid JSON, retrying...")
      attempts+=1

  if len(records) < count:
    raise RuntimeError(f'could only generate {len(records)} unique records')

  synthetic_data =  records[:count]

  df = pd.DataFrame(synthetic_data)
  df.to_csv("synthetic_data.csv", index=False)
  csv_path = "synthetic_data.csv"

  return df.head(10), csv_path

In [13]:
def gradio_generate(domain, description, count):

  print("GRADIO FUNCTION CALLED")
  print(domain)
  print(description)
  print(count)

  start_time = time.time()

  file_preview, csv_path = generate_records(
      tokenizer=tokenizer, model=model,
      domain=domain, description=description, count=int(count))

  elapsed = round((time.time() - start_time) / 60, 2)  # returns seconds so div by 60, 2 for decimal places.

  status = (
      f'Dataset generated successfully.\n'
      f'Requested count : {count}\n'
      f'Generation time : {elapsed} minutes'
  )

  return status, file_preview, csv_path

In [ ]:
with gr.Blocks(title="Synthetic Data Generator") as gradio_ui:

  gr.Markdown(
      """
      # Synthetic Dataset Generator

      Generate synthetic datasets using Llama 3.1 8B.

      Generation may take several minutes.

      Estimated times:

      25 records ≈ 3 minutes
      50 records ≈ 5 minutes
      100 records ≈ 10 minutes
      200 records ≈ 20 minutes
      """
  )

  with gr.Row():

    with gr.Column():

      domain_input = gr.Textbox(label="Domain", placeholder="Healthcare, Education, Finance, Corporate...")

      description_input = gr.Textbox(label="Dataset Description", lines=6,
                placeholder=
                """
                Generate employee records with:
                name
                age
                salary
                joining date
                department
                """
            )
      count_input = gr.Dropdown(choices=[25,50,75,100,125,150,175,200], value=25, label="Number of Records")

      generate_button = gr.Button("Generate Dataset", variant="primary")


    with gr.Column():

      status_output = gr.Textbox(label="Status", interactive=False)

      preview_output = gr.Dataframe(label="Dataset Preview")

      download_output = gr.File(label="Download CSV")


  generate_button.click(
      fn=gradio_generate,
      inputs=[domain_input, description_input, count_input],
      outputs=[status_output, preview_output, download_output]
  )

gradio_ui.launch(debug=True)